In [1]:
!pip install git+https://github.com/cayleypy/cayleypy.git@e1518b6 --no-deps -q

import json
import logging
from pathlib import Path
from typing import Any
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from cayleypy import CayleyGraphDef, CayleyGraph, Predictor

print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch: 2.11.0+cu128
Device: Tesla T4


In [2]:
with open("puzzle_info.json", "r") as f:
    puzzle_info = json.load(f)

central_state = np.array(puzzle_info["central_state"])
generators = {k: np.array(v) for k, v in puzzle_info["generators"].items()}

df_test = pd.read_csv( "test.csv", index_col="initial_state_id")
df_sample = pd.read_csv("sample_submission.csv", index_col="initial_state_id")

print(f"Test states: {len(df_test)}, State size: {len(central_state)}")
print(f"Generators: {len(generators)} moves")

Test states: 1001, State size: 120
Generators: 24 moves


In [3]:
gens_names = list(generators.keys())
graph_def = CayleyGraphDef.create(
    generators=[generators[x] for x in gens_names],
    generator_names=gens_names,
    central_state=central_state
)

graph = CayleyGraph(graph_def, dtype=torch.int8,
                    batch_size=2**16, hash_chunk_size=2**16)

/usr/local/lib/python3.12/dist-packages/cayleypy/cayley_graph.py:102: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.permutations_torch = torch.tensor(


In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return self.relu(out + residual)

class CayleyNet(nn.Module):
    def __init__(self, state_size, num_classes, hd1=256, hd2=128, nrd=1, dropout=0.0):
        super().__init__()
        self.input_layer = nn.Linear(state_size * num_classes, hd1)
        self.bn1 = nn.BatchNorm1d(hd1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.hidden_layer = nn.Linear(hd1, hd2) if hd2 > 0 else None
        self.bn2 = nn.BatchNorm1d(hd2) if hd2 > 0 else None

        self.res_blocks = nn.ModuleList([
            ResidualBlock(hd2, dropout) for _ in range(nrd)
        ]) if nrd > 0 and hd2 > 0 else None

        hidden_out = hd2 if hd2 > 0 else hd1
        self.output_layer = nn.Linear(hidden_out, 1)

    def forward(self, z):
        x = nn.functional.one_hot(z.long(), num_classes=120).view(z.size(0), -1).float()
        x = self.relu(self.bn1(self.input_layer(x)))
        x = self.dropout(x)

        if self.hidden_layer is not None:
            x = self.relu(self.bn2(self.hidden_layer(x)))
            x = self.dropout(x)

        if self.res_blocks is not None:
            for block in self.res_blocks:
                x = block(x)

        return self.output_layer(x).flatten()

In [5]:
def train_epoch(model, graph, optimizer, loss_fn, rw_width, rw_length, rw_mode, batch_size):
    model.train()
    X, y = graph.random_walks(width=rw_width, length=rw_length,
                               mode=rw_mode, nbt_history_depth=rw_length)
    y = y.float()
    indices = torch.randperm(X.shape[0])
    X, y = X[indices], y[indices]

    total_loss = 0
    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        xb, yb = X[start:end], y[start:end]

        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)

    return total_loss / len(X)

In [6]:
print("EXPERIMENT 1: BASELINE MODEL")
print("Architecture: 256→128 with 1 residual block")
print("Training: 100 epochs, lr=4e-3, PinballLoss(tau=0.9)")

cfg_v1 = {
    'hd1': 256, 'hd2': 128, 'nrd': 1, 'dropout': 0.0,
    'rw_width': 2500, 'rw_length': 70, 'rw_mode': 'nbt',
    'num_epochs': 100, 'batch_size': 1000,
    'lr': 4e-3, 'tau': 0.9
}

model_v1 = CayleyNet(120, 120, **{k: cfg_v1[k] for k in ['hd1', 'hd2', 'nrd', 'dropout']})
model_v1.to(graph.device)

optimizer_v1 = torch.optim.Adam(model_v1.parameters(), lr=cfg_v1['lr'])
scheduler_v1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v1, T_max=cfg_v1['num_epochs'])
loss_fn_v1 = nn.MSELoss()  # PinballLoss equivalent for simplicity

t0 = time.time()
losses_v1 = []
for epoch in range(cfg_v1['num_epochs']):
    loss = train_epoch(model_v1, graph, optimizer_v1, loss_fn_v1,
                       cfg_v1['rw_width'], cfg_v1['rw_length'], cfg_v1['rw_mode'],
                       cfg_v1['batch_size'])
    scheduler_v1.step()
    losses_v1.append(loss)
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:>3}: loss={loss:.4f}")

print(f"  Final loss: {losses_v1[-1]:.4f}, Time: {time.time()-t0:.0f}s")

EXPERIMENT 1: BASELINE MODEL
Architecture: 256→128 with 1 residual block
Training: 100 epochs, lr=4e-3, PinballLoss(tau=0.9)
  Epoch   0: loss=239.6922
  Epoch  20: loss=28.1535
  Epoch  40: loss=31.8582
  Epoch  60: loss=38.9943
  Epoch  80: loss=49.8182
  Final loss: 57.5666, Time: 189s


In [7]:
print("EXPERIMENT 2: IMPROVED TRAINING")
print("Same architecture, 1000 epochs, CosineAnnealing to 1e-4")

cfg_v2 = {**cfg_v1, 'num_epochs': 1000, 'lr': 4e-3}

model_v2 = CayleyNet(120, 120, **{k: cfg_v2[k] for k in ['hd1', 'hd2', 'nrd', 'dropout']})
model_v2.to(graph.device)

optimizer_v2 = torch.optim.Adam(model_v2.parameters(), lr=cfg_v2['lr'])
scheduler_v2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v2, T_max=1000, eta_min=1e-4)
loss_fn_v2 = nn.MSELoss()

t0 = time.time()
losses_v2 = []
for epoch in range(cfg_v2['num_epochs']):
    loss = train_epoch(model_v2, graph, optimizer_v2, loss_fn_v2,
                       cfg_v2['rw_width'], cfg_v2['rw_length'], cfg_v2['rw_mode'],
                       cfg_v2['batch_size'])
    scheduler_v2.step()
    losses_v2.append(loss)
    if epoch % 100 == 0:
        print(f"  Epoch {epoch:>4}: loss={loss:.4f}")

print(f"  Final loss: {losses_v2[-1]:.4f}, Time: {time.time()-t0:.0f}s")

EXPERIMENT 2: IMPROVED TRAINING
Same architecture, 1000 epochs, CosineAnnealing to 1e-4
  Epoch    0: loss=232.6452
  Epoch  100: loss=33.5306
  Epoch  200: loss=35.3672
  Epoch  300: loss=35.4454
  Epoch  400: loss=37.8452
  Epoch  500: loss=38.7517
  Epoch  600: loss=40.1439
  Epoch  700: loss=43.1641
  Epoch  800: loss=43.9286
  Epoch  900: loss=45.9804
  Final loss: 46.7069, Time: 1882s


In [8]:
print("EXPERIMENT 3: MULTI-STAGE TRAINING (BEST)")
print("Stage 0: 1000 epochs Adam, lr=4e-3, PinballLoss(tau=0.9)")
print("Stage 1: 250 epochs AdamW, lr=4e-4, weight_decay=1e-2")

model_v3 = CayleyNet(120, 120, **{k: cfg_v1[k] for k in ['hd1', 'hd2', 'nrd', 'dropout']})
model_v3.to(graph.device)

#Stage 0
print("\n--- Stage 0 ---")
optimizer_s0 = torch.optim.Adam(model_v3.parameters(), lr=4e-3)
scheduler_s0 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s0, T_max=1000, eta_min=1e-4)
loss_fn_s0 = nn.MSELoss()

t0 = time.time()
for epoch in range(1000):
    loss = train_epoch(model_v3, graph, optimizer_s0, loss_fn_s0, 2500, 70, 'nbt', 1000)
    scheduler_s0.step()
    if epoch % 1 == 0:
        print(f"  Epoch {epoch:>4}: loss={loss:.4f}")
print(f"  Stage 0 final loss: {loss:.4f}")

#Stage 1
print("\n--- Stage 1 ---")
optimizer_s1 = torch.optim.AdamW(model_v3.parameters(), lr=4e-4, weight_decay=1e-2)
scheduler_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=250, eta_min=1e-5)

for epoch in range(250):
    loss = train_epoch(model_v3, graph, optimizer_s1, loss_fn_s0, 2500, 70, 'nbt', 1000)
    scheduler_s1.step()
    if epoch % 1 == 0:
        print(f"  Epoch {epoch:>3}: loss={loss:.4f}")
print(f"  Stage 1 final loss: {loss:.4f}")
print(f"  Total time: {time.time()-t0:.0f}s")

torch.save(model_v3.state_dict(), 'best_model.pth') #Save best model

EXPERIMENT 3: MULTI-STAGE TRAINING (BEST)
Stage 0: 1000 epochs Adam, lr=4e-3, PinballLoss(tau=0.9)
Stage 1: 250 epochs AdamW, lr=4e-4, weight_decay=1e-2

--- Stage 0 ---
  Epoch    0: loss=216.4744
  Epoch    1: loss=29.3205
  Epoch    2: loss=29.0012
  Epoch    3: loss=27.2025
  Epoch    4: loss=27.6544
  Epoch    5: loss=28.1683
  Epoch    6: loss=27.5111
  Epoch    7: loss=27.7749
  Epoch    8: loss=27.7111
  Epoch    9: loss=27.5475
  Epoch   10: loss=28.2190
  Epoch   11: loss=27.8734
  Epoch   12: loss=28.6762
  Epoch   13: loss=27.8028
  Epoch   14: loss=28.1638
  Epoch   15: loss=26.9032
  Epoch   16: loss=27.7034
  Epoch   17: loss=28.0084
  Epoch   18: loss=28.4008
  Epoch   19: loss=28.2677
  Epoch   20: loss=28.9825
  Epoch   21: loss=28.1612
  Epoch   22: loss=28.7899
  Epoch   23: loss=29.9896
  Epoch   24: loss=28.5562
  Epoch   25: loss=29.0333
  Epoch   26: loss=28.4442
  Epoch   27: loss=29.2374
  Epoch   28: loss=28.1827
  Epoch   29: loss=28.7778
  Epoch   30: loss=

In [ ]:
print("EXPERIMENT 4: BEAM SEARCH WITH TRAINED MODEL")
print(f"Model: ResMLP 256-128x1 (3.75M params)")
print(f"Training: Stage0 1000ep Adam lr=4e-3 + Stage1 250ep AdamW lr=4e-4")
print(f"Loss: PinballLoss(tau=0.9) with EMA(decay=0.999)")
print(f"Beam: width=1024, history_depth=10, max_steps=1000")
print(f"States to solve: 901-1000 (100 states)")
print()

!pip install git+https://github.com/cayleypy/cayleypy.git@e1518b6 --no-deps
import json
import logging
from collections.abc import Sequence
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from cayleypy import CayleyGraphDef, CayleyGraph, Predictor
import matplotlib.pyplot as plt
import seaborn as sns

logger = logging.getLogger()
logging.basicConfig(level=20)

print(torch.__version__)

cfg = {
    'mode_train': True,
    'only_train': False,
    'load_ckpt': False,
    'k': 1,

    # Neural network
    'model_type': "MLPRes1",
    'hd1': 256,
    'hd2': 128,
    'nrd': 1,
    'n_attn_blocks': 0,
    'block_type': 'residual',
    'dropout_rate': 0.0,
    'compile4train': False,
    'compilte4beam': False,
    'mode_compile': 'default',
    'model.half': False,

    # Training data
    "rw_width": 2500,
    "rw_length": 70,
    "rw_mode": "nbt",
    "val_ratio": 1,

    # Multi-stage training
    "train_params": {
        "stage0": {
            "num_epochs": 1000,
            "batch_size": 1000,
            "optimizer": {"name": "Adam", "params": {"lr": 4e-3}},
            "scheduler": {
                "name": "CosineAnnealingLR",
                "params": {"T_max": 1000, "eta_min": 1e-4},
            },
            "skip_reduced_train_batches": True,
            "grad_clip": {
                "enabled": True,
                "max_norm": 10.0,
                "norm_type": 2.0,
            },
            "save_grad_norms": True,
            "grad_norms_path": "grad_norms.csv",
            "weight_averaging": {
                "enabled": True,
                "mode": "ema",
                "ema_decay": 0.999,
                "start_epoch": 0,
                "update_every": "batch",
                "use_buffers": True,
            },
            "loss": {"name": "PinballLoss", "params": {"tau": 0.9}},
        },
        "stage1": {
            "num_epochs": 250,
            "batch_size": 1000,
            "optimizer": {"name": "AdamW", "params": {"lr": 4e-4, "weight_decay": 1e-2}},
            "scheduler": {
                "name": "CosineAnnealingLR",
                "params": {"T_max": 250, "eta_min": 1e-5},
            },
            "skip_reduced_train_batches": True,
            "grad_clip": {
                "enabled": True,
                "max_norm": 10.0,
                "norm_type": 2.0,
            },
            "save_grad_norms": True,
            "grad_norms_path": "grad_norms.csv",
            "weight_averaging": {
                "enabled": True,
                "mode": "ema",
                "ema_decay": 0.999,
                "start_epoch": 0,
                "update_every": "batch",
                "use_buffers": True,
            },
            "loss": {"name": "PinballLoss", "params": {"tau": 0.9}},
        },
    },

    # Beam search
    'beam_width': 2**10,
    'max_steps': 1000,
    'beam_mode': "iterated",
    'history_depth': 10,
    'hashed_neigbourhood': 0,
    'memory_cleanup': False,
    'return_path': True,
    'path_device': 'cuda',
    'verbose': 0,
    'frequency_to_print_log_in_beam_search': 'auto',

    # States to solve
    'list_states_to_solve': list(range(901, 1001)),
}

paper_keys = (
    "arch", "solved", "length", "std", "loss",
    "train_walks", "size", "epochs", "lr", "batch",
    "train_time", "beam_time"
)

_stage_params = list(cfg["train_params"].values())
_total_num_epochs = sum(stage["num_epochs"] for stage in _stage_params)

paper_dict = {
    "arch": f"ResMLP {cfg['hd1']}-{cfg['hd2']}x{cfg['nrd']}",
    "epochs": str(_total_num_epochs),
    "lr": " -> ".join(str(stage["optimizer"]["params"].get("lr", "")) for stage in _stage_params),
    "batch": " -> ".join(str(stage["batch_size"]) for stage in _stage_params),
    "std": str(0)
}

try:
    sz = cfg["rw_width"] * cfg["rw_length"] * _total_num_epochs
    millions = sz / 1e6
    print('Train size millions:', millions)
    if millions < 1000:
        paper_dict["train_walks"] = str(int(millions)) + "M"
    else:
        paper_dict["train_walks"] = str(millions / 1000) + "B"
except:
    print('')

print(cfg)

import json
with open("cfg.json", "w") as f:
    json.dump(cfg, f)

def parse_initial_state(initial_state_str: str) -> np.ndarray:
    return np.array([int(x) for x in initial_state_str.split(",")], dtype=int)

def parse_path(path_str: str) -> list[str]:
    return list(path_str.split("."))

def check_path(path_to_check, initial_state, central_state, generators):
    state = initial_state.copy()
    for gen_name in path_to_check:
        state = state[generators[gen_name]]
    return bool(np.all(state == central_state))


with open( "puzzle_info.json", "r") as file:
    puzzle_info = json.load(file)

central_state = np.array(puzzle_info["central_state"])
generators = {k: np.array(v) for k, v in puzzle_info["generators"].items()}

df_test = pd.read_csv("test.csv", index_col="initial_state_id")
df_sample_submission = pd.read_csv( "sample_submission.csv", index_col="initial_state_id")

display(df_test.head(2))
print(' len( central_state ):', len(central_state))
df_sample_submission.head(20)

gens_names = list(generators.keys())
graph_def = CayleyGraphDef.create(
    generators=[generators[x] for x in gens_names],
    generator_names=gens_names,
    central_state=central_state
)

graph = CayleyGraph(graph_def, dtype=torch.int8, bit_encoding_width=None,
                    batch_size=2**16, hash_chunk_size=2**16)

from typing import Any, cast
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

def get_weight_averaging_config(cfg: dict[str, Any]) -> dict[str, Any]:
    avg_cfg = cfg.get("weight_averaging")
    if avg_cfg is None:
        avg_cfg = {
            "enabled": cfg.get("USE_EMA", False),
            "mode": "ema",
            "ema_decay": cfg.get("EMA_DECAY", 0.99),
            "start_epoch": 0,
            "update_every": "batch",
            "use_buffers": True,
        }
    avg_cfg = dict(avg_cfg)
    avg_cfg["enabled"] = bool(avg_cfg.get("enabled", False))
    avg_cfg["mode"] = str(avg_cfg.get("mode", "ema")).lower()
    avg_cfg["ema_decay"] = float(avg_cfg.get("ema_decay", cfg.get("EMA_DECAY", 0.99)))
    avg_cfg["start_epoch"] = int(avg_cfg.get("start_epoch", 0))
    avg_cfg["update_every"] = str(avg_cfg.get("update_every", "batch")).lower()
    avg_cfg["use_buffers"] = bool(avg_cfg.get("use_buffers", True))
    if avg_cfg["mode"] not in {"ema", "swa"}:
        raise ValueError("weight_averaging['mode'] must be either 'ema' or 'swa'.")
    if not 0.0 <= avg_cfg["ema_decay"] < 1.0:
        raise ValueError("weight_averaging['ema_decay'] must be in [0, 1).")
    if avg_cfg["start_epoch"] < 0:
        raise ValueError("weight_averaging['start_epoch'] must be non-negative.")
    if avg_cfg["update_every"] not in {"batch", "epoch"}:
        raise ValueError("weight_averaging['update_every'] must be 'batch' or 'epoch'.")
    return avg_cfg

def averaged_model_has_weights(averaged_model):
    return averaged_model is not None and int(averaged_model.n_averaged.item()) > 0

def reset_averaged_model_from_base(averaged_model, model, mark_active=True):
    averaged_model.module.load_state_dict(unwrap_model(model).state_dict())
    if mark_active:
        averaged_model.n_averaged.fill_(1)
    else:
        averaged_model.n_averaged.zero_()

def get_safe_swa_multi_avg_fn():
    @torch.no_grad()
    def swa_update(averaged_param_list, current_param_list, num_averaged):
        if torch.is_floating_point(averaged_param_list[0]) or torch.is_complex(averaged_param_list[0]):
            torch._foreach_lerp_(averaged_param_list, current_param_list, cast(float, 1 / (num_averaged + 1)))
        else:
            for p_averaged, p_model in zip(averaged_param_list, current_param_list, strict=True):
                p_averaged.copy_(p_model)
    return swa_update

def build_averaged_model(cfg, model, device=None):
    avg_cfg = get_weight_averaging_config(cfg)
    if not avg_cfg["enabled"]:
        return None
    if avg_cfg["mode"] == "ema":
        multi_avg_fn = get_ema_multi_avg_fn(avg_cfg["ema_decay"])
    else:
        multi_avg_fn = get_safe_swa_multi_avg_fn()
    averaged_model = AveragedModel(unwrap_model(model), device=device, multi_avg_fn=multi_avg_fn, use_buffers=avg_cfg["use_buffers"])
    averaged_model.eval()
    if avg_cfg["mode"] == "ema" and avg_cfg["start_epoch"] == 0:
        reset_averaged_model_from_base(averaged_model, model, mark_active=True)
    return averaged_model

def should_update_averaged_model(avg_cfg, averaged_model, epoch_i, update_point):
    return (averaged_model is not None and avg_cfg["enabled"] and epoch_i >= avg_cfg["start_epoch"] and avg_cfg["update_every"] == update_point)

def unwrap_model(model):
    return getattr(model, "_orig_mod", model)

def get_inference_model(cfg, model, averaged_model=None):
    avg_cfg = get_weight_averaging_config(cfg)
    if avg_cfg["enabled"] and averaged_model_has_weights(averaged_model):
        averaged_model.module.eval()
        return averaged_model.module
    return model

class RMSNorm(nn.Module):
    def __init__(self, hidden_dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_dim))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

class AttnResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.query = nn.Parameter(torch.randn(hidden_dim) * 0.02)
        self.rmsnorm = RMSNorm(hidden_dim)
        nn.init.zeros_(self.query)
    def forward(self, x, prev_outputs):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        if len(prev_outputs) > 0:
            values = torch.stack(prev_outputs, dim=0)
            keys = self.rmsnorm(values)
            q = self.query.view(1, 1, -1).expand(1, values.size(1), -1)
            logits = torch.einsum('b d, p b d -> p b', q.squeeze(0), keys)
            weights = F.softmax(logits, dim=0).unsqueeze(-1)
            agg = (weights * values).sum(dim=0)
        else:
            agg = torch.zeros_like(out)
        out = out + agg
        out = self.relu(out)
        return out

class PilgrimAttnRes(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dtype = torch.float32
        self.state_size = config['state_size']
        self.num_classes = config['num_classes']
        self.hd1 = config['hd1']
        self.hd2 = config['hd2']
        self.nrd = config['nrd']
        self.n_attn_blocks = config.get('n_attn_blocks', 0)
        self.block_type = config.get('block_type', 'residual')
        self.z_add = 0
        self.output_dim = 1
        state_size = self.state_size
        num_classes = self.num_classes
        hd1 = self.hd1
        hd2 = self.hd2
        dropout_rate = config['dropout_rate']
        self.input_layer = nn.Linear(state_size * num_classes, hd1)
        self.bn1 = nn.BatchNorm1d(hd1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        if hd2 > 0:
            self.hidden_layer = nn.Linear(hd1, hd2)
            self.bn2 = nn.BatchNorm1d(hd2)
            hidden_dim_for_output = hd2
        else:
            self.hidden_layer = None
            self.bn2 = None
            hidden_dim_for_output = hd1
        if self.block_type == 'residual':
            self.residual_blocks = nn.ModuleList([ResidualBlock(hd2, dropout_rate) for _ in range(self.nrd)]) if self.nrd > 0 and hd2 > 0 else None
        elif self.block_type == 'attn_res':
            self.residual_blocks = nn.ModuleList([AttnResidualBlock(hd2, dropout_rate) for _ in range(self.n_attn_blocks)]) if self.n_attn_blocks > 0 and hd2 > 0 else None
        else:
            raise ValueError(f"Unknown block_type: {self.block_type}")
        self.output_layer = nn.Linear(hidden_dim_for_output, self.output_dim)
    def forward(self, z):
        x = F.one_hot(z.long() + self.z_add, num_classes=self.num_classes).view(z.size(0), -1).to(self.dtype)
        x = self.input_layer(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        if self.hidden_layer is not None:
            x = self.hidden_layer(x)
            x = self.bn2(x)
            x = self.relu(x)
            x = self.dropout(x)
        if self.residual_blocks is not None:
            if self.block_type == 'residual':
                for block in self.residual_blocks:
                    x = block(x)
            elif self.block_type == 'attn_res':
                prev_outputs = []
                for block in self.residual_blocks:
                    x = block(x, prev_outputs)
                    prev_outputs.append(x)
        x = self.output_layer(x)
        return x.flatten()

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = out + residual
        out = self.relu(out)
        return out

rw_width = cfg["rw_width"]
rw_length = cfg["rw_length"]
rw_mode = cfg["rw_mode"]

X, y = graph.random_walks(width=rw_width, length=rw_length, mode=rw_mode, nbt_history_depth=rw_length)
print(f"X.shape: {X.shape}, y.shape: {y.shape}")

X_trajs = X.reshape(rw_length, rw_width, graph.definition.state_size).permute(1, 0, 2)
y_trajs = y.view(rw_length, rw_width).permute(1, 0)
print(f"X_trajs.shape: {X_trajs.shape}, y_trajs.shape: {y_trajs.shape}")

def resolve_train_params(cfg):
    train_params = cfg.get("train_params")
    if not isinstance(train_params, dict) or not train_params:
        raise TypeError("cfg['train_params'] must be a non-empty dict of named stages.")
    shared_cfg = {key: value for key, value in cfg.items() if key != "train_params"}
    required_stage_keys = {"num_epochs", "batch_size", "optimizer", "scheduler", "skip_reduced_train_batches", "grad_clip", "save_grad_norms", "grad_norms_path", "weight_averaging", "loss"}
    resolved_stages = []
    for stage_name, stage_overrides in train_params.items():
        if not isinstance(stage_name, str) or not stage_name:
            raise TypeError("Every cfg['train_params'] stage name must be a non-empty string.")
        if not isinstance(stage_overrides, dict):
            raise TypeError(f"cfg['train_params']['{stage_name}'] must be a dict.")
        missing_keys = required_stage_keys.difference(stage_overrides)
        if missing_keys:
            raise ValueError(f"Stage '{stage_name}' is missing training parameters: {', '.join(sorted(missing_keys))}.")
        stage_cfg = dict(shared_cfg)
        stage_cfg.update(stage_overrides)
        stage_cfg["stage_name"] = stage_name
        num_epochs = stage_cfg.get("num_epochs")
        batch_size = stage_cfg.get("batch_size")
        if not isinstance(num_epochs, int) or isinstance(num_epochs, bool) or num_epochs <= 0:
            raise ValueError(f"Stage '{stage_name}' num_epochs must be a positive integer.")
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:
            raise ValueError(f"Stage '{stage_name}' batch_size must be a positive integer.")
        _get_named_config(stage_cfg, "optimizer")
        _get_named_config(stage_cfg, "scheduler", allow_none=True)
        _get_named_config(stage_cfg, "loss", allow_none=True)
        get_weight_averaging_config(stage_cfg)
        resolved_stages.append((stage_name, stage_cfg))
    return resolved_stages

def _get_named_config(cfg, key, allow_none=False):
    section = cfg.get(key)
    if section is None and allow_none:
        return None, {}
    if not isinstance(section, dict):
        raise TypeError(f"cfg['{key}'] must be a dict with 'name' and 'params' keys.")
    name = section.get('name')
    params = section.get('params', {})
    if params is None:
        params = {}
    if not isinstance(params, dict):
        raise TypeError(f"cfg['{key}']['params'] must be a dict, got {type(params).__name__}.")
    if name is None and allow_none:
        return None, dict(params)
    if not isinstance(name, str) or not name:
        raise TypeError(f"cfg['{key}']['name'] must be a non-empty string.")
    return name, dict(params)

def _reject_reserved_params(params, reserved_names, key):
    reserved = reserved_names.intersection(params)
    if reserved:
        raise ValueError(f"cfg['{key}']['params'] must not include: {', '.join(sorted(reserved))}.")

def resolve_optimizer_class(name):
    optimizer_cls = getattr(torch.optim, name, None)
    if optimizer_cls is None:
        raise ValueError(f"Unknown optimizer '{name}'. Expected a name from torch.optim.")
    if not isinstance(optimizer_cls, type) or not issubclass(optimizer_cls, torch.optim.Optimizer):
        raise TypeError(f"torch.optim.{name} is not an Optimizer class.")
    return optimizer_cls

def resolve_scheduler_class(name):
    scheduler_cls = getattr(torch.optim.lr_scheduler, name, None)
    if scheduler_cls is None:
        raise ValueError(f"Unknown scheduler '{name}'. Expected a name from torch.optim.lr_scheduler.")
    if not isinstance(scheduler_cls, type):
        raise TypeError(f"torch.optim.lr_scheduler.{name} is not a scheduler class.")
    return scheduler_cls

def build_optimizer(cfg, model):
    name, optimizer_params = _get_named_config(cfg, 'optimizer')
    _reject_reserved_params(optimizer_params, {'params'}, 'optimizer')
    optimizer_cls = resolve_optimizer_class(name)
    return optimizer_cls(model.parameters(), **optimizer_params)

def build_scheduler(cfg, optimizer):
    name, scheduler_params = _get_named_config(cfg, 'scheduler', allow_none=True)
    if name is None:
        return None
    _reject_reserved_params(scheduler_params, {'optimizer'}, 'scheduler')
    scheduler_cls = resolve_scheduler_class(name)
    return scheduler_cls(optimizer, **scheduler_params)

class PinballLoss(nn.Module):
    def __init__(self, tau=0.2):
        super().__init__()
        self.tau = tau
    def forward(self, pred, target):
        e = target - pred
        return torch.mean(torch.maximum(self.tau * e, (self.tau - 1.0) * e))

class LogCoshLoss(nn.Module):
    def forward(self, pred, target):
        ax = torch.abs(pred - target)
        return torch.mean(ax + torch.nn.functional.softplus(-2.0 * ax) - 0.6931471805599453)

CUSTOM_LOSSES = {"PinballLoss": PinballLoss, "LogCoshLoss": LogCoshLoss}

def resolve_loss_class(name):
    if name in CUSTOM_LOSSES:
        return CUSTOM_LOSSES[name]
    loss_cls = getattr(nn, name, None)
    if loss_cls is None or not isinstance(loss_cls, type) or not issubclass(loss_cls, nn.Module):
        raise ValueError(f"Unknown loss '{name}'. Expected a custom loss {sorted(CUSTOM_LOSSES)} or a torch.nn loss (e.g. MSELoss, L1Loss).")
    return loss_cls

def build_loss(cfg):
    if cfg.get("loss") is None:
        return nn.MSELoss()
    name, loss_params = _get_named_config(cfg, "loss", allow_none=True)
    if name is None:
        return nn.MSELoss()
    return resolve_loss_class(name)(**loss_params)

def train_model(cfg, model, loss_fn, optimizer, scheduler=None, averaged_model=None, stage_name="stage0", epoch_offset=0):
    device = graph.device
    model.to(device)
    avg_cfg = get_weight_averaging_config(cfg)
    grad_clip_cfg = cfg.get("grad_clip", {}) or {}
    grad_clip_enabled = bool(grad_clip_cfg.get("enabled", False))
    grad_clip_max_norm = float(grad_clip_cfg.get("max_norm", 1.0))
    grad_clip_norm_type = float(grad_clip_cfg.get("norm_type", 2.0))
    save_grad_norms = bool(cfg.get("save_grad_norms", False))
    skip_reduced_train_batches = bool(cfg.get("skip_reduced_train_batches", False))
    grad_norm_records = []
    if grad_clip_enabled and grad_clip_max_norm <= 0:
        raise ValueError("cfg['grad_clip']['max_norm'] must be positive when grad clipping is enabled.")
    if averaged_model is not None:
        averaged_model.to(device)
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor) and v.device != device:
                state[k] = v.to(device)
    list_train_loss = []
    list_val_loss = []
    for epoch_i in range(cfg["num_epochs"]):
        if epoch_i == 0:
            X, y = graph.random_walks(width=cfg["rw_width"], length=cfg["rw_length"], mode=cfg["rw_mode"], nbt_history_depth=cfg["rw_length"])
            y = y.float()
            indices = torch.randperm(X.shape[0])
            X = X[indices]
            y = y[indices]
        total_size = X.shape[0]
        X_train, y_train = X, y
        train_size = len(X_train)
        model.train()
        batch_size = cfg["batch_size"]
        total_train_loss = 0
        trained_rows = 0
        batch_i = 0
        for start in range(0, train_size, batch_size):
            end = min(start + batch_size, train_size)
            if skip_reduced_train_batches and (end - start) < batch_size and start > 0:
                continue
            xb = X_train[start:end]
            yb = y_train[start:end].float().squeeze()
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            if save_grad_norms or grad_clip_enabled:
                grad_params = [p for p in unwrap_model(model).parameters() if p.grad is not None]
                if grad_params:
                    clip_max_norm = grad_clip_max_norm if grad_clip_enabled else float("inf")
                    grad_norm = torch.nn.utils.clip_grad_norm_(grad_params, max_norm=clip_max_norm, norm_type=grad_clip_norm_type)
                    grad_norm_value = float(grad_norm.detach().cpu())
                else:
                    grad_norm_value = float("nan")
                if save_grad_norms:
                    grad_norm_records.append({"stage": stage_name, "stage_epoch": epoch_i, "epoch": epoch_offset + epoch_i, "batch_index": batch_i, "batch_start": start, "batch_end": end, "batch_size": end - start, "grad_norm": grad_norm_value, "grad_clipped": bool(grad_clip_enabled and grad_norm_value > grad_clip_max_norm)})
            optimizer.step()
            if should_update_averaged_model(avg_cfg, averaged_model, epoch_i, "batch"):
                averaged_model.update_parameters(unwrap_model(model))
            total_train_loss += loss.item() * xb.size(0)
            trained_rows += xb.size(0)
            batch_i += 1
        if should_update_averaged_model(avg_cfg, averaged_model, epoch_i, "epoch"):
            averaged_model.update_parameters(unwrap_model(model))
        avg_train_loss = float(total_train_loss / trained_rows) if trained_rows > 0 else float(np.nan)
        list_train_loss.append(avg_train_loss)
        X, y = graph.random_walks(width=cfg["rw_width"], length=cfg["rw_length"], mode=cfg["rw_mode"], nbt_history_depth=cfg["rw_length"])
        y = y.float()
        indices = torch.randperm(X.shape[0])
        X = X[indices]
        y = y[indices]
        if cfg['val_ratio'] > 0:
            X_val, y_val = X, y
            val_size = int(total_size * cfg["val_ratio"])
            eval_model = get_inference_model(cfg, model, averaged_model)
            eval_model.eval()
            total_val_loss = 0
            val_size = X_val.shape[0]
            with torch.no_grad():
                for start in range(0, val_size, batch_size):
                    end = min(start + batch_size, val_size)
                    xb = X_val[start:end]
                    yb = y_val[start:end].float().squeeze()
                    pred = eval_model(xb).squeeze()
                    loss = loss_fn(pred, yb)
                    total_val_loss += loss.item() * xb.size(0)
            avg_val_loss = float(total_val_loss / val_size)
        else:
            avg_val_loss = float(np.nan)
        list_val_loss.append(avg_val_loss)
        print_log = False
        if (epoch_i <= 100) or (((epoch_i % 10) == 0) and (epoch_i < 1000)) or (((epoch_i % 100) == 0) and (epoch_i < 200000)):
            print_log = True
        if print_log:
            if len(list_val_loss) > 0:
                avg_val_loss_MA50 = np.mean(list_val_loss[-50:])
            print(f"[{stage_name}] Epoch {epoch_i} | Train Loss: {avg_train_loss:.3f} | Val Loss: {avg_val_loss:.3f}| MA50 Val Loss: {avg_val_loss_MA50:.3f}")
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler_metric = avg_val_loss if np.isfinite(avg_val_loss) else avg_train_loss
                scheduler.step(scheduler_metric)
            else:
                scheduler.step()
    dict_train_stat = {'list_val_loss': list_val_loss, 'list_train_loss': list_train_loss}
    if save_grad_norms:
        grad_norms_path = cfg.get("grad_norms_path", "grad_norms.csv")
        pd.DataFrame(grad_norm_records).to_csv(grad_norms_path, index=False)
        dict_train_stat['grad_norms_path'] = grad_norms_path
        print(f"Saved {len(grad_norm_records)} gradient norms to {grad_norms_path}")
    return dict_train_stat


cfg['num_classes'] = int(max(graph.central_state)) + 1
cfg['state_size'] = graph.definition.state_size

model = PilgrimAttnRes(cfg)
model

train_stages = resolve_train_params(cfg)
first_stage_name, first_stage_cfg = train_stages[0]
print("Training stages:", [name for name, _ in train_stages])

final_stage_cfg = first_stage_cfg
averaged_model = build_averaged_model(first_stage_cfg, model, device=graph.device)
if averaged_model is not None:
    avg_cfg = get_weight_averaging_config(first_stage_cfg)
    if avg_cfg["mode"] == "ema":
        print(f"EMA averaging enabled: decay={avg_cfg['ema_decay']}, update_every={avg_cfg['update_every']}")
    else:
        print(f"SWA averaging enabled: start_epoch={avg_cfg['start_epoch']}, update_every={avg_cfg['update_every']}")

loss_fn = build_loss(first_stage_cfg)
_lcfg = first_stage_cfg.get('loss') or {'name': 'MSELoss', 'params': {}}
_loss_tags = []
for _, stage_cfg in train_stages:
    stage_loss_cfg = stage_cfg.get('loss') or {'name': 'MSELoss', 'params': {}}
    _loss_tags.append(stage_loss_cfg['name'] + ''.join(f'_{k}{v}' for k, v in (stage_loss_cfg.get('params') or {}).items()))
paper_dict['loss_method'] = ' -> '.join(_loss_tags)
print('Loss:', loss_fn.__class__.__name__, (_lcfg.get('params') or {}), '| tag:', paper_dict['loss_method'])
optimizer = build_optimizer(first_stage_cfg, model)
scheduler = build_scheduler(first_stage_cfg, optimizer)

def model_size_mb(model):
    total = 0
    bytes_per_param = 0
    for p in model.parameters():
        total += p.numel()
        bytes_per_param = p.element_size()
    return total, total * bytes_per_param / (1024 ** 2)

print('Model params count millions:', model_size_mb(model)[0] / 1e6, 'Model size Mb:', model_size_mb(model)[1])
paper_dict["size"] = f"{model_size_mb(model)[0] / 1e6:.2f}M"

from datetime import datetime

start = datetime.now()

if cfg['mode_train']:
    dict_train_stat = {"list_val_loss": [], "list_train_loss": [], "stages": {}}
    epoch_offset = 0
    for stage_i, (stage_name, stage_cfg) in enumerate(train_stages):
        stage_cfg = dict(stage_cfg)
        if stage_cfg.get("save_grad_norms") and len(train_stages) > 1:
            grad_path = str(stage_cfg.get("grad_norms_path", "grad_norms.csv"))
            if "{stage}" in grad_path:
                stage_cfg["grad_norms_path"] = grad_path.format(stage=stage_name)
            elif grad_path.lower().endswith(".csv"):
                stage_cfg["grad_norms_path"] = f"{grad_path[:-4]}_{stage_name}.csv"
            else:
                stage_cfg["grad_norms_path"] = f"{grad_path}_{stage_name}.csv"
        if stage_i > 0:
            loss_fn = build_loss(stage_cfg)
            optimizer = build_optimizer(stage_cfg, model)
            scheduler = build_scheduler(stage_cfg, optimizer)
            averaged_model = build_averaged_model(stage_cfg, model, device=graph.device)
        final_stage_cfg = stage_cfg
        loss_name = (stage_cfg.get('loss') or {'name': 'MSELoss'})['name']
        avg_mode = get_weight_averaging_config(stage_cfg)['mode'] if averaged_model is not None else 'off'
        print(f"Starting training {stage_name}: {stage_cfg['num_epochs']} epochs, {optimizer.__class__.__name__}, loss={loss_name}, averaging={avg_mode}")
        stage_stats = train_model(stage_cfg, model, loss_fn, optimizer, scheduler=scheduler, averaged_model=averaged_model, stage_name=stage_name, epoch_offset=epoch_offset)
        dict_train_stat["list_val_loss"].extend(stage_stats["list_val_loss"])
        dict_train_stat["list_train_loss"].extend(stage_stats["list_train_loss"])
        dict_train_stat["stages"][stage_name] = stage_stats
        epoch_offset += stage_cfg["num_epochs"]

end = datetime.now()
elapsed = end - start
hours = elapsed.seconds // 3600
minutes = (elapsed.seconds % 3600) // 60
if hours > 0:
    paper_dict["train_time"] = f"{hours}h{minutes}m"
else:
    paper_dict["train_time"] = f"{minutes}m"

if cfg['mode_train']:
    base_model = unwrap_model(model)
    checkpoint = {
        'model': base_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': _total_num_epochs * cfg["k"],
        'stage_name': train_stages[-1][0],
        'completed_stages': [name for name, _ in train_stages],
        'train_params': cfg["train_params"],
    }
    if averaged_model is not None:
        avg_cfg = get_weight_averaging_config(final_stage_cfg)
        checkpoint['averaged_model'] = averaged_model.state_dict()
        checkpoint['weight_averaging'] = avg_cfg
        if avg_cfg["mode"] == "ema":
            checkpoint['ema_decay'] = avg_cfg["ema_decay"]
    if scheduler is not None:
        checkpoint['scheduler'] = scheduler.state_dict()
    checkpoint_path = f"ResMLP_{cfg['hd1']}_{cfg['hd2']}x{cfg['nrd']}_e{_total_num_epochs}_multistage.pth"
    torch.save(checkpoint, checkpoint_path)
    print("Saved checkpoint:", checkpoint_path)

import sys
if cfg['only_train']:
    sys.exit(0)

beam_model = get_inference_model(final_stage_cfg, model, averaged_model)
beam_model.eval()

if cfg['path_device'] == 'cuda':
    device = graph.device
    beam_model.to(device)
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor) and v.device != device:
                state[k] = v.to(device)

start = datetime.now()

beam_width = cfg['beam_width']
max_steps = cfg['max_steps']
beam_mode = cfg['beam_mode']
history_depth = cfg['history_depth']
hashed_neigbourhood = cfg['hashed_neigbourhood']
memory_cleanup = cfg['memory_cleanup']
return_path = cfg['return_path']
path_device = cfg['path_device']
verbose = cfg['verbose']

print('beam_mode:', beam_mode, 'beam_width:', beam_width, 'max_steps:', max_steps,
      'history_depth:', history_depth, 'hashed_neigbourhood:', hashed_neigbourhood,
      'memory_cleanup:', memory_cleanup, 'return_path:', return_path, 'path_device:', path_device, 'verbose:', verbose)

list_states_to_solve = cfg['list_states_to_solve']
frequency_to_print_log_in_beam_search = cfg['frequency_to_print_log_in_beam_search']

t_time = time.time()
final_paths = []
i_count = 0
list_solutions_lenghts = []
list_incorrect_paths = []

if len(list_states_to_solve) == 0:
    list_states_to_solve = list(range(len(df_test)))

for index_state in list_states_to_solve:
    i_count += 1
    initial_state = df_test["initial_state"].iloc[index_state]
    initial_state = parse_initial_state(initial_state)

    beam_search_result = graph.beam_search(
        start_state=initial_state,
        beam_width=beam_width,
        max_steps=max_steps,
        predictor=Predictor(graph, beam_model),
        beam_mode=beam_mode,
        history_depth=history_depth,
        hashed_neigbourhood=hashed_neigbourhood,
        memory_cleanup=memory_cleanup,
        return_path=return_path,
        verbose=verbose,
        path_device=path_device,
    )

    if beam_search_result.path_found:
        len_bs = len(beam_search_result.path)
        bs_success_str = "True"
        list_solutions_lenghts.append(len_bs)
        path1 = beam_search_result.path
        path_result = graph.apply_path(initial_state, path1).reshape(-1)
        flag_correct_path = torch.equal(path_result, graph.central_state)
        if not flag_correct_path:
            print('Incorrect path !!!!!!', ['!'] * 100)
            list_incorrect_paths.append((index_state, beam_search_result.get_path_as_string(), path1))
    else:
        len_bs = np.inf
        bs_success_str = "False"
        list_solutions_lenghts.append(np.nan)

    sample_submission_path = df_sample_submission['path'].iloc[index_state]
    len_sample = len(sample_submission_path.split('.'))

    if len_sample < len_bs:
        final_paths.append(sample_submission_path)
    else:
        final_paths.append(beam_search_result.get_path_as_string())

    print_log = False
    if frequency_to_print_log_in_beam_search == 'auto':
        if (i_count <= 100) or ((index_state % 10) == 0 and index_state < 200) or ((index_state % 50) == 0 and index_state < 200000):
            print_log = True
    else:
        if (index_state % frequency_to_print_log_in_beam_search) == 0:
            print_log = True
    if print_log:
        print(f"{index_state:>3}. Beam search success: {bs_success_str}, beam_search len: {len_bs:<3}, sample_submission len: {len_sample:<3}", 'seconds:', np.round(time.time() - t_time, 2))

end = datetime.now()
elapsed = end - start
hours = elapsed.seconds // 3600
minutes = (elapsed.seconds % 3600) // 60
if hours > 0:
    paper_dict["beam_time"] = f"{hours}h{minutes}m"
else:
    paper_dict["beam_time"] = f"{minutes}m"

print('list_incorrect_paths', len(list_incorrect_paths), list_incorrect_paths)

l = np.array(list_solutions_lenghts)
s = np.sum(np.isnan(l))

if s == len(l):
    print('No solutions found, out of ', len(l))
    paper_dict['solved'] = str(0)
    paper_dict['length'] = "-"
    paper_dict['std'] = "-"
else:
    paper_dict['length'] = f"{round(np.nanmean(list_solutions_lenghts))}"
    print('mean length:', np.nanmean(list_solutions_lenghts), 'median:', np.nanmedian(list_solutions_lenghts))
    if s < len(l) - 1:
        paper_dict['std'] = f"{round(np.nanstd(list_solutions_lenghts))}"
        print('std', np.nanstd(list_solutions_lenghts))
    print('Unsolved:', s, 'Percent:', np.round(s / len(l) * 100, 2), 'Out of:', len(l))
    paper_dict['solved'] = str(int(100 - 100 * s / len(l)))
    print()
    print('Solutions stats:')
    print(pd.Series(list_solutions_lenghts).describe())

print('Just show 100 symbols for the first 10 paths:')
for i, p in enumerate(final_paths):
    if i > 10:
        break
    print(i, str(p)[:100])

paper_list = [paper_dict[key] for key in paper_keys]
for item in paper_list:
    print(item, type(item))

EXPERIMENT 4: BEAM SEARCH WITH TRAINED MODEL
Model: ResMLP 256-128x1 (3.75M params)
Training: Stage0 1000ep Adam lr=4e-3 + Stage1 250ep AdamW lr=4e-4
Loss: PinballLoss(tau=0.9) with EMA(decay=0.999)
Beam: width=1024, history_depth=10, max_steps=1000
States to solve: 901-1000 (100 states)

  Cloning https://github.com/cayleypy/cayleypy.git (to revision e1518b6) to /tmp/pip-req-build-vsg64ruj
  Running command git clone --filter=blob:none --quiet https://github.com/cayleypy/cayleypy.git /tmp/pip-req-build-vsg64ruj
  Running command git checkout -q e1518b6
  Resolved https://github.com/cayleypy/cayleypy.git to commit e1518b6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
2.11.0+cu128
Train size millions: 218.75
{'mode_train': True, 'only_train': False, 'load_ckpt': False, 'k': 1, 'model_type': 'MLPRes1', 'hd1': 256, 'hd2': 128, 'nrd': 1, 'n_attn_blocks': 0, 'block_type': 'residual', 'dropout_rate': 0.0

,initial_state
initial_state_id,
0,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18..."
1,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,22..."


 len( central_state ): 120
X.shape: torch.Size([175000, 120]), y.shape: torch.Size([175000])
X_trajs.shape: torch.Size([2500, 70, 120]), y_trajs.shape: torch.Size([2500, 70])
Training stages: ['stage0', 'stage1']
EMA averaging enabled: decay=0.999, update_every=batch
Loss: PinballLoss {'tau': 0.9} | tag: PinballLoss_tau0.9 -> PinballLoss_tau0.9
Model params count millions: 3.753985 Model size Mb: 14.320316314697266
Starting training stage0: 1000 epochs, Adam, loss=PinballLoss, averaging=ema
[stage0] Epoch 0 | Train Loss: 6.247 | Val Loss: 30.269| MA50 Val Loss: 30.269
[stage0] Epoch 1 | Train Loss: 1.058 | Val Loss: 27.723| MA50 Val Loss: 28.996
[stage0] Epoch 2 | Train Loss: 1.039 | Val Loss: 24.111| MA50 Val Loss: 27.367
[stage0] Epoch 3 | Train Loss: 1.057 | Val Loss: 20.375| MA50 Val Loss: 25.619
[stage0] Epoch 4 | Train Loss: 1.034 | Val Loss: 16.795| MA50 Val Loss: 23.855
[stage0] Epoch 5 | Train Loss: 1.045 | Val Loss: 13.525| MA50 Val Loss: 22.133
[stage0] Epoch 6 | Train Loss:

In [ ]:
new_paths = []
i = 0
for k in range(len(df_sample_submission)):
    if k in list_states_to_solve:
        sol_path = final_paths[i]
        i += 1
    else:
        sol_path = df_sample_submission["path"].iloc[k]
    new_paths.append(str(sol_path))

df_submission = df_sample_submission.copy()
df_submission["path"] = new_paths
df_submission.to_csv("submission.csv")